## Parallel Workflow with MAF and Microsoft Foundry

![parallel-workflow](./Assets/parallel-workflow.png)

In [ ]:
%pip install -U agent-framework-core agent-framework-foundry azure-ai-projects azure-identity python-dotenv ipython

### Setting up the Environment

In [3]:
import os
from dotenv import load_dotenv
from agent_framework import Executor, WorkflowContext, handler

load_dotenv()
project_endpoint = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
model = os.getenv("AI_FOUNDRY_DEPLOYMENT_NAME")

print("Project Endpoint: ", project_endpoint)
print("Model: ", model)

Project Endpoint:  https://udemo-103-resource.services.ai.azure.com/api/projects/udemo-103
Model:  gpt-5.2


### Defining a Function to Create a Chat Agent

In [4]:
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential
from agent_framework import Agent

# Cell 2: Define async workflow
async def create_agent(agent_name: str,
                       agent_instructions: str) -> Agent:
    
    # Create async Azure credential
    credential = DefaultAzureCredential()

    # creating the Foundry Project Client
    chat_client = FoundryChatClient(
        project_endpoint=project_endpoint,
        model=model,
        credential=credential,
    )
    
    agent = Agent(
            name=agent_name,
            instructions=agent_instructions,
            client=chat_client,
        )

    print("{} Agent created successfully!".format(agent_name))
    return agent

### Creating the Location Picker Agent

In [5]:
location_picker_agent = await create_agent(
    agent_name = "Location-Picker-Agent",
    agent_instructions = """You are a helpful assistant that helps users pick a location for their vacation."""
)

Location-Picker-Agent Agent created successfully!


### Creating the Destination Recommender Agent

In [6]:
destination_recommender_agent = await create_agent(
    agent_name = "Destination-Recommender-Agent",
    agent_instructions = """You are a travel expert that provides personalized vacation recommendations based on user preferences and locations. """
)

Destination-Recommender-Agent Agent created successfully!


### Creating the Weather Agent

In [7]:
weather_agent = await create_agent(
    agent_name = "Weather-Agent",
    agent_instructions = """You are a weather expert that provides accurate and up-to-date weather information for various locations selected"""
)

Weather-Agent Agent created successfully!


### Creating the Cuisine Suggestion Agent

In [8]:
cuisine_suggestion_agent = await create_agent(
    agent_name = "Cuisine-Suggestion-Agent",
    agent_instructions = """You are a culinary expert that suggests popular local cuisines and dining options based on the selected vacation destinations."""
)

Cuisine-Suggestion-Agent Agent created successfully!


### Creating the Itinerary Planner Agent

In [9]:
itinerary_planner_agent = await create_agent(
    agent_name = "Itinerary-Planner-Agent",
    agent_instructions = """You are an itinerary planning expert that creates detailed travel itineraries based on user preferences, selected destinations, weather conditions, and local cuisine options."""
)

Itinerary-Planner-Agent Agent created successfully!


### Creating the Location Selector Agent Executor

In [10]:
class LocationSelectorExecutor(Executor):

    @handler
    async def handle(self, user_query: str, ctx: WorkflowContext[str]) -> None:
        response = await location_picker_agent.run(user_query)

        await ctx.send_message(str(response))

### Creating the Destination Recommender Agent Executor

In [11]:


class DestinationRecommenderExecutor(Executor):

    @handler
    async def handle(self, location: str, ctx: WorkflowContext[str]) -> None:
        response = await destination_recommender_agent.run(location)

        await ctx.send_message(str(response))

### Creating the Weather Agent Executor

In [12]:
class WeatherExecutor(Executor):

    @handler
    async def handle(self, location: str, ctx: WorkflowContext[str]) -> None:
        response = await weather_agent.run(location)

        await ctx.send_message(str(response))

### Creating the Cuisine Suggestion Agent Executor

In [13]:
class CuisineSuggestionExecutor(Executor):

    @handler
    async def handle(self, location: str, ctx: WorkflowContext[str]) -> None:
        response = await cuisine_suggestion_agent.run(location)

        await ctx.send_message(str(response))

### Creating the Itinerary Planner Agent Executor

In [14]:
class ItineraryPlannerExecutor(Executor):

    @handler
    async def handle(self, results: list[str], ctx: WorkflowContext[str]) -> None:
        response = await itinerary_planner_agent.run(results)

        await ctx.yield_output(str(response))

### Building the Parallel Workflow

In [ ]:
# Create the executor instances / objects
location_selector_executor = LocationSelectorExecutor(id="LocationSelector")
destination_recommender_executor = DestinationRecommenderExecutor(id="DestinationRecommender")
weather_executor = WeatherExecutor(id="Weather")
cuisine_suggestion_executor = CuisineSuggestionExecutor(id="CuisineSuggestion")
itinerary_planner_executor = ItineraryPlannerExecutor(id="ItineraryPlanner")
from agent_framework import WorkflowBuilder, WorkflowViz

# Build the workflow
workflow = (
    WorkflowBuilder(start_executor=location_selector_executor)
    .add_fan_out_edges(location_selector_executor, [destination_recommender_executor, weather_executor, cuisine_suggestion_executor])
    .add_fan_in_edges([destination_recommender_executor, weather_executor, cuisine_suggestion_executor], itinerary_planner_executor)
    .build()
)

viz = WorkflowViz(workflow)


### Generating Mermaid Diagram for Visualization

In [23]:
from IPython.display import Markdown, display, Image
import base64
mermaid_content = viz.to_mermaid()
# printing mermaid content as markdown
display(Markdown(f"``` mermaid\n{mermaid_content}\n```"))
encoded = base64.urlsafe_b64encode(mermaid_content.encode()).decode()
display(Image(url=f"https://mermaid.ink/img/{encoded}"))

``` mermaid
flowchart TD
  LocationSelector["LocationSelector (Start)"];
  DestinationRecommender["DestinationRecommender"];
  Weather["Weather"];
  CuisineSuggestion["CuisineSuggestion"];
  ItineraryPlanner["ItineraryPlanner"];
  fan_in__ItineraryPlanner__ca5ba06d((fan-in))
  CuisineSuggestion --> fan_in__ItineraryPlanner__ca5ba06d;
  DestinationRecommender --> fan_in__ItineraryPlanner__ca5ba06d;
  Weather --> fan_in__ItineraryPlanner__ca5ba06d;
  fan_in__ItineraryPlanner__ca5ba06d --> ItineraryPlanner;
  LocationSelector --> DestinationRecommender;
  LocationSelector --> Weather;
  LocationSelector --> CuisineSuggestion;
```

### Running the Workflow and Streaming Events

In [17]:
# Run the workflow and stream events in notebook

async def main():
    prompt = "help me plan a vacation to iraq with the following details: I love historical sites, prefer warm weather, and enjoy trying local foods."

    result = await workflow.run(prompt)

    print("Workflow completed with result:")
    outputs = result.get_outputs()

    for output in outputs:
        print(output)

await main()



Workflow completed with result:
Got it. I can turn your cuisine notes into a **bookable, day-by-day “what to eat where” plan**, but I need one missing piece: you mention choosing **Option A or B**, yet only the **two templates** are shown. To avoid guessing, please reply with:

1) **A or B** = which template you mean (or paste the two options).  
2) **Likely month** (even a rough one, e.g., “late Oct” or “March”).  
3) **Any hard constraints**: dietary restrictions, alcohol yes/no, and whether you’ll have a **driver** (important for meal timing outside cities).

In the meantime, here are **two ready-to-use “what to eat where” day-by-day overlays** matched to your earlier templates (you can book around these immediately; I’ll refine once you confirm A/B + month).

---

## Overlay for Template 1 (9 days): Kurdistan + Baghdad + Babylon + Ur + Marshes

### Days 1–3: **Erbil**
**Best meals to plan**
- **Breakfast:** light—tea + bread + eggs (save appetite for grills later).
- **Lunch:** baz